## Example Ring and Sidechain Datasets

In [61]:
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from pathlib import Path

# Define the directory where the files will be saved
HOME_DIR = Path.cwd()  # Change this if needed
DATA_DIR = HOME_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)  # Ensure the directory exists

# Create Rings Dataset
rings_data = [
    {"id": "R1", "smiles": "A", "n_subs": 2},  # Benzene with 2 attachment points
    {"id": "R2", "smiles": "B", "n_subs": 3},  # Pyridine with 3 attachment points
    {"id": "R3", "smiles": "C", "n_subs": 1},  # Pyridine with 1 attachment point
    {"id": "R3", "smiles": "D", "n_subs": 5},  # Pyridine with 1 attachment point
    {"id": "R3", "smiles": "E", "n_subs": 4},  # Pyridine with 1 attachment point
    {"id": "R3", "smiles": "F", "n_subs": 4},  # Pyridine with 1 attachment point
    {"id": "R3", "smiles": "G", "n_subs": 4},  # Pyridine with 1 attachment point
    {"id": "R3", "smiles": "H", "n_subs": 4},  # Pyridine with 1 attachment point
    {"id": "R3", "smiles": "I", "n_subs": 4},  # Pyridine with 1 attachment point
    {"id": "R3", "smiles": "J", "n_subs": 4},  # Pyridine with 1 attachment point
]

# Convert to DataFrame
df_rings = pd.DataFrame(rings_data)

# Save as Parquet
df_rings.to_parquet(DATA_DIR / "ring_list.parquet", engine="pyarrow")

# Create Sidechains Dataset
sidechains_data = [
    {"id": "S1", "smiles": "WE"},   # Methoxy (-OCH3)
    {"id": "S2", "smiles": "ARE"},  # Chloromethyl (-CH2Cl)
    {"id": "S2", "smiles": "SO"},  # Chloromethyl (-CH2Cl)
    {"id": "S2", "smiles": "BACK"},  # Chloromethyl (-CH2Cl)
    {"id": "S2", "smiles": "NOT"},  # Chloromethyl (-CH2Cl)
    {"id": "S2", "smiles": "YET"},  # Chloromethyl (-CH2Cl)
    {"id": "S2", "smiles": "TILL"},  # Chloromethyl (-CH2Cl)
    {"id": "S2", "smiles": "THAT"},  # Chloromethyl (-CH2Cl)
    {"id": "S2", "smiles": "LIST"},  # Chloromethyl (-CH2Cl)
    {"id": "S2", "smiles": "OUT"},  # Chloromethyl (-CH2Cl)
    {"id": "S2", "smiles": "STEIM"},  # Chloromethyl (-CH2Cl)
    {"id": "S2", "smiles": "YO"},  # Chloromethyl (-CH2Cl)
    {"id": "S2", "smiles": "AYO"},  # Chloromethyl (-CH2Cl)
    {"id": "S2", "smiles": "PLS"},  # Chloromethyl (-CH2Cl)
    {"id": "S2", "smiles": "FINISH"},  # Chloromethyl (-CH2Cl)
]

# Convert to DataFrame
df_sidechains = pd.DataFrame(sidechains_data)

# Save as Parquet
df_sidechains.to_parquet(DATA_DIR / "sidechain_list.parquet", engine="pyarrow")

print(f"✅ Rings and Sidechains datasets saved in {DATA_DIR}")


✅ Rings and Sidechains datasets saved in C:\Users\zachg\PycharmProjects\BillionMolecules\billion_mol_db\testing_recombination\data


## Recombination Logic

Original logic that parses parquet files incorrectly

In [1]:
import duckdb
import pandas as pd
import concurrent.futures
import pyarrow as pa
from pathlib import Path
import pyarrow.parquet as pq
import itertools
import os
import shutil


def new_combination(ring_smiles, sub_smiles_list):
    """Generates a new combined SMILES string from a ring and multiple sidechains."""
    return ring_smiles + "(" + ")(".join(sub_smiles_list) + ")"


# Function to process and write each batch to its own file
def process_and_write_batch(df_batch, parquet_schema, batch_id, temp_dir):
    df_batch["combination"] = df_batch.apply(
        lambda row: new_combination(row["ring_smiles"], [row[f"sub{i}_smiles"] for i in range(1, row["n_subs"] + 1)]),
        axis=1,
    )

    df_batch = df_batch.drop(columns=["n_subs"])  # Remove n_subs column after processing

    # Convert DataFrame to Apache Arrow table
    table = pa.Table.from_pandas(df_batch, schema=parquet_schema)

    # Write to a temporary Parquet file
    temp_file = os.path.join(temp_dir, f"batch_{batch_id}.parquet")
    pq.write_table(table, temp_file)


def main():
    # Variable and file paths
    HOME_DIR = Path.cwd()  # Gets the current working directory
    # HOME_DIR = Path(__file__).resolve().parent.parent
    ring_file = HOME_DIR / "data" / "ring_list.parquet"
    sidechain_file = HOME_DIR / "data" / "sidechain_list.parquet"
    mol_db_file = HOME_DIR / "data" / "mol_db.parquet"
    temp_dir = HOME_DIR / "temp_batches"  # Directory to store temporary batch files
    temp_dir.mkdir(exist_ok=True)

    id_name = "id"
    smiles_name = "smiles"
    n_subs_name = "n_subs"
    batch_size = 1000  # Adjust this batch size as necessary

    # Load the molecule fragments from a file into DuckDB
    con = duckdb.connect()
    con.execute(f"CREATE TABLE rings AS SELECT * FROM '{ring_file}'")  # Load rings into DuckDB
    con.execute(f"CREATE TABLE sidechains AS SELECT * FROM '{sidechain_file}'")  # Load sidechains into DuckDB

    # Set up Parquet schema dynamically based on the maximum number of substitutions
    max_subs = con.execute(f"SELECT MAX({n_subs_name}) FROM rings").fetchone()[0]
    parquet_schema_fields = [
        ("ring_id", pa.string()),
        ("ring_smiles", pa.string()),
        ("n_subs", pa.int32()),  # Keep track of how many substitutions
    ]
    for i in range(1, max_subs + 1):
        parquet_schema_fields.append((f"sub{i}_id", pa.string()))
        parquet_schema_fields.append((f"sub{i}_smiles", pa.string()))
    parquet_schema_fields.append(("combination", pa.string()))  # Final combined SMILES
    parquet_schema = pa.schema(parquet_schema_fields)

    # Create the base query for rings and the first sidechain (must be an INNER JOIN to guarantee at least one sidechain)
    query = f"""
        SELECT rings.{id_name} AS ring_id, rings.{smiles_name} AS ring_smiles, rings.{n_subs_name} AS n_subs,
               side1.{id_name} AS sub1_id, side1.{smiles_name} AS sub1_smiles
        FROM rings
        INNER JOIN sidechains AS side1 ON TRUE
        WHERE rings.{n_subs_name} >= 1
        """
    
    # Dynamically add additional LEFT JOINs for more sidechains based on `n_subs`
    for i in range(2, max_subs + 1):
        query += f"""
        LEFT JOIN sidechains AS side{i} ON rings.{n_subs_name} >= {i}
        """
    
    print("Generated SQL Query:\n", query)  # Debugging: Check query before execution

    # Execute the query and use fetch_df_chunk to fetch data in chunks
    cursor = con.execute(query)

    executor = concurrent.futures.ProcessPoolExecutor()
    batch_id = 0

    while True:
        # Fetch the next chunk of data using fetch_df_chunk()
        df_batch = cursor.fetch_df_chunk(batch_size)
        if df_batch.empty:
            break  # No more data

        # Process each batch in parallel and write to its own file
        future = executor.submit(
            process_and_write_batch, df_batch=df_batch, parquet_schema=parquet_schema, batch_id=batch_id, temp_dir=temp_dir
        )

        batch_id += 1

    # After processing, merge all temp files into the final file
    with pq.ParquetWriter(mol_db_file, parquet_schema) as writer:
        for temp_file in temp_dir.glob("*.parquet"):
            table = pq.read_table(temp_file)
            writer.write_table(table)

    total_rows = con.execute(f"SELECT COUNT(*) FROM ({query})").fetchone()[0]
    print(f"Mol DB with {total_rows} rows written to {mol_db_file}.")

    # Cleanup temporary files and connections
    shutil.rmtree(temp_dir)  # Remove entire temp directory
    con.close()
    executor.shutdown()


if __name__ == "__main__":
    main()


Generated SQL Query:
 
        SELECT rings.id AS ring_id, rings.smiles AS ring_smiles, rings.n_subs AS n_subs,
               side1.id AS sub1_id, side1.smiles AS sub1_smiles
        FROM rings
        INNER JOIN sidechains AS side1 ON TRUE
        WHERE rings.n_subs >= 1
        
        LEFT JOIN sidechains AS side2 ON rings.n_subs >= 2
        
        LEFT JOIN sidechains AS side3 ON rings.n_subs >= 3
        


ParserException: Parser Error: syntax error at or near "LEFT"

New start from scratch logic ensuring we can use duckdb to load the data from parquet files correctly

In [18]:
import duckdb
import pandas as pd
from pathlib import Path

# Define file paths
HOME_DIR = Path.cwd()
ring_file = HOME_DIR / "data" / "ring_list.parquet"
sidechain_file = HOME_DIR / "data" / "sidechain_list.parquet"
combinations_file = HOME_DIR / "data" / "combinations_2.parquet"

# Connect to DuckDB
con = duckdb.connect()

# Load Parquet files into DuckDB tables
con.execute(f"CREATE TABLE rings AS SELECT * FROM '{ring_file}'")
con.execute(f"CREATE TABLE sidechains AS SELECT * FROM '{sidechain_file}'")
con.execute(f"CREATE TABLE combinations AS SELECT * FROM '{combinations_file}'")

# Query the tables and load results into Pandas DataFrames
df_rings = con.execute("SELECT * FROM rings").fetchdf()
df_sidechains = con.execute("SELECT * FROM sidechains").fetchdf()
df_combinations = con.execute("SELECT * FROM combinations").fetchdf()

# Print the DataFrames
print("Rings DataFrame:")
print(df_rings)

print("\nSidechains DataFrame:")
print(df_sidechains)

print("\nCombinations DataFrame:")
print(df_combinations)

# Close the DuckDB connection
con.close()


IOException: IO Error: No files found that match the pattern "C:\Users\zachg\PycharmProjects\BillionMolecules\billion_mol_db\testing_recombination\data\combinations_2.parquet"

Script to just read the two input files, and generate an empty combinations dataset

In [5]:
import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

class GenComs_Ring_Sidechains:
    def __init__(self, ring_file, sidechain_file, output_file):
        """
        Initializes the class with input ring and sidechain datasets and the output file path.
        """
        self.ring_file = ring_file
        self.sidechain_file = sidechain_file
        self.output_file = output_file
        self.con = duckdb.connect()

    def load_data(self):
        """
        Loads ring and sidechain datasets into DuckDB tables.
        """
        self.con.execute(f"CREATE TABLE rings AS SELECT * FROM '{self.ring_file}'")
        self.con.execute(f"CREATE TABLE sidechains AS SELECT * FROM '{self.sidechain_file}'")

    def create_empty_parquet(self):
        """
        Creates an empty Parquet file with the correct schema but no data.
        """
        # Define the expected schema based on ring-sidechain combination structure
        parquet_schema = pa.schema([
            ("ring_id", pa.string()),
            ("ring_smiles", pa.string()),
            ("sub1_id", pa.string()),
            ("sub1_smiles", pa.string()),
            ("sub2_id", pa.string()),  # Keeping space for at least 2 sidechains
            ("sub2_smiles", pa.string())
        ])

        # Create an empty Arrow Table
        empty_table = pa.Table.from_pandas(pd.DataFrame(columns=parquet_schema.names))

        # Write the empty table to a Parquet file
        pq.write_table(empty_table, self.output_file)

        print(f"Empty Parquet file created: {self.output_file}")

    def close_connection(self):
        """
        Closes the DuckDB connection.
        """
        self.con.close()

# Example usage
if __name__ == "__main__":
    # Define file paths
    HOME_DIR = Path.cwd()
    ring_file = HOME_DIR / "data" / "ring_list.parquet"
    sidechain_file = HOME_DIR / "data" / "sidechain_list.parquet"
    output_file = HOME_DIR / "data" / "combinations.parquet"

    # Initialize and run the class
    generator = GenComs_Ring_Sidechains(ring_file, sidechain_file, output_file)
    generator.load_data()
    generator.create_empty_parquet()
    generator.close_connection()


Empty Parquet file created: C:\Users\zachg\PycharmProjects\BillionMolecules\billion_mol_db\testing_recombination\data\combinations.parquet


In [9]:
import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import itertools
import os

class GenComs_Ring_Sidechains:
    def __init__(self, ring_file, sidechain_file, output_file):
        """
        Initializes the class with input ring and sidechain datasets and the output file path.
        """
        self.ring_file = ring_file
        self.sidechain_file = sidechain_file
        self.output_file = output_file
        self.con = duckdb.connect()

    def load_data(self):
        """
        Loads ring and sidechain datasets into DuckDB tables.
        """
        self.con.execute(f"CREATE TABLE rings AS SELECT * FROM '{self.ring_file}'")
        self.con.execute(f"CREATE TABLE sidechains AS SELECT * FROM '{self.sidechain_file}'")

    def ensure_valid_parquet(self):
        """
        Ensures that the output Parquet file is valid by creating an empty file with the correct schema if needed.
        """
        parquet_schema = pa.schema([
            ("ring_id", pa.string()),
            ("ring_smiles", pa.string()),
            ("sub1_id", pa.string()),
            ("sub1_smiles", pa.string()),
            ("sub2_id", pa.string()),  # Ensure consistent string type
            ("sub2_smiles", pa.string()),  # Ensure consistent string type
            ("combination", pa.string())
        ])

        # If the file does not exist or is empty, create a valid empty Parquet file
        if not os.path.exists(self.output_file) or os.path.getsize(self.output_file) == 0:
            print(f"Initializing {self.output_file} with an empty valid schema.")
            empty_table = pa.Table.from_pandas(pd.DataFrame(columns=parquet_schema.names), schema=parquet_schema)
            pq.write_table(empty_table, self.output_file)

    def generate_combinations(self):
        """
        Processes rings one at a time, generating all valid combinations with sidechains,
        and saves them iteratively to `combinations.parquet`.
        """
        # Ensure output file is valid before writing
        self.ensure_valid_parquet()

        # Read rings table into DuckDB
        rings_df = self.con.execute("SELECT * FROM rings").fetchdf()
        
        # Read sidechains table into DuckDB
        sidechains_df = self.con.execute("SELECT * FROM sidechains").fetchdf()

        # Process one ring at a time to prevent memory overload
        for _, ring in rings_df.iterrows():
            ring_id = ring["id"]
            ring_smiles = ring["smiles"]
            n_subs = ring["n_subs"]

            # Generate all possible combinations of `n_subs` sidechains
            sidechain_combinations = list(itertools.combinations(sidechains_df.itertuples(index=False), n_subs))

            results = []
            for sidechain_set in sidechain_combinations:
                sub_ids = [sc.id for sc in sidechain_set]
                sub_smiles = [sc.smiles for sc in sidechain_set]

                # Create the new combination SMILES string
                combination_smiles = ring_smiles + "(" + ")(".join(sub_smiles) + ")"

                # Construct a row, ensuring all fields are explicitly strings
                row = {
                    "ring_id": str(ring_id),
                    "ring_smiles": str(ring_smiles),
                    "sub1_id": str(sub_ids[0]) if len(sub_ids) > 0 else "",
                    "sub1_smiles": str(sub_smiles[0]) if len(sub_smiles) > 0 else "",
                    "sub2_id": str(sub_ids[1]) if len(sub_ids) > 1 else "",
                    "sub2_smiles": str(sub_smiles[1]) if len(sub_smiles) > 1 else "",
                    "combination": str(combination_smiles)
                }
                results.append(row)

            # Convert results to a DataFrame with explicit types
            df_combinations = pd.DataFrame(results, columns=["ring_id", "ring_smiles", "sub1_id", "sub1_smiles",
                                                             "sub2_id", "sub2_smiles", "combination"])
            df_combinations = df_combinations.astype(str)  # Ensure consistent string types

            # Append to Parquet file manually
            if not df_combinations.empty:
                # Read the existing Parquet file
                existing_table = pq.read_table(self.output_file)

                # Convert DataFrame to Arrow Table with the same schema
                new_table = pa.Table.from_pandas(df_combinations, schema=existing_table.schema)

                # Concatenate existing and new tables
                combined_table = pa.concat_tables([existing_table, new_table])

                # Write back to Parquet file
                pq.write_table(combined_table, self.output_file)

            print(f"Processed ring {ring_id} with {n_subs} substituent positions.")

    def close_connection(self):
        """
        Closes the DuckDB connection.
        """
        self.con.close()

# Example usage
if __name__ == "__main__":
    # Define file paths
    HOME_DIR = Path.cwd()
    ring_file = HOME_DIR / "data" / "ring_list.parquet"
    sidechain_file = HOME_DIR / "data" / "sidechain_list.parquet"
    output_file = HOME_DIR / "data" / "combinations.parquet"

    # Initialize and run the class
    generator = GenComs_Ring_Sidechains(ring_file, sidechain_file, output_file)
    generator.load_data()
    generator.generate_combinations()
    generator.close_connection()


Processed ring R1 with 2 substituent positions.
Processed ring R2 with 3 substituent positions.
Processed ring R3 with 1 substituent positions.


In [64]:
import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import itertools
import os
import numpy as np

class GenComs_Ring_Sidechains:
    def __init__(self, ring_file, sidechain_file, output_dir):
        """
        Initializes the class with input ring and sidechain datasets and the output directory.
        The Parquet file name will be automatically determined to avoid overwrites.
        """
        self.ring_file = ring_file
        self.sidechain_file = sidechain_file
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.output_file = self.get_unique_filename()
        self.con = duckdb.connect()

    def get_unique_filename(self):
        """
        Generate a unique Parquet filename by appending _1, _2, etc., if a file already exists.
        """
        base_filename = "combinations"
        extension = ".parquet"
        output_path = self.output_dir / f"{base_filename}{extension}"

        counter = 1
        while output_path.exists():
            output_path = self.output_dir / f"{base_filename}_{counter}{extension}"
            counter += 1

        return output_path

    def load_data(self):
        """
        Loads ring and sidechain datasets into DuckDB tables.
        """
        self.con.execute(f"CREATE TABLE rings AS SELECT * FROM '{self.ring_file}'")
        self.con.execute(f"CREATE TABLE sidechains AS SELECT * FROM '{self.sidechain_file}'")

    def ensure_valid_parquet(self):
        """
        Ensures that the output Parquet file is valid by creating an empty file with the correct schema if needed.
        """
        parquet_schema = pa.schema([
            ("ring_id", pa.string()),
            ("ring_smiles", pa.string()),
            ("sub_id", pa.list_(pa.string())),  # Ordered list of sidechain IDs
            ("sub_smiles", pa.list_(pa.string())),  # Ordered list of sidechain SMILES
            ("combination", pa.string())  # Placeholder for now
        ])

        # If the file does not exist or is empty, create a valid empty Parquet file
        if not os.path.exists(self.output_file) or os.path.getsize(self.output_file) == 0:
            print(f"Initializing {self.output_file} with an empty valid schema.")
            empty_table = pa.Table.from_pandas(pd.DataFrame(columns=parquet_schema.names), schema=parquet_schema)
            pq.write_table(empty_table, self.output_file)

    def generate_combinations(self):
        """
        Processes rings one at a time, generating all possible sidechain combinations
        (including repeated sidechains) based on the number of substitution points.
        Saves results to the Parquet file iteratively.
        """
        # Ensure output file is valid before writing
        self.ensure_valid_parquet()

        # Read rings and sidechains tables into DuckDB
        rings_df = self.con.execute("SELECT * FROM rings").fetchdf()
        sidechains_df = self.con.execute("SELECT * FROM sidechains").fetchdf()

        # Define the output Parquet schema
        parquet_schema = pa.schema([
            ("ring_id", pa.string()),
            ("ring_smiles", pa.string()),
            ("sub_id", pa.list_(pa.string())),  # Ordered list of sidechain IDs
            ("sub_smiles", pa.list_(pa.string())),  # Ordered list of sidechain SMILES
            ("combination", pa.string())  # Placeholder for now
        ])

        # Process one ring at a time
        for _, ring in rings_df.iterrows():
            ring_id = ring["id"]
            ring_smiles = ring["smiles"]
            n_subs = ring["n_subs"]

            # Generate all possible combinations of `n_subs` sidechains (including repetitions)
            sidechain_combinations = list(itertools.product(sidechains_df.itertuples(index=False), repeat=n_subs))

            results = []
            for sidechain_set in sidechain_combinations:
                sub_ids = [sc.id for sc in sidechain_set]
                sub_smiles = [sc.smiles for sc in sidechain_set]

                # Construct a row with ordered lists
                row = {
                    "ring_id": str(ring_id),
                    "ring_smiles": str(ring_smiles),
                    "sub_id": sub_ids,  # List of sidechain IDs
                    "sub_smiles": sub_smiles,  # List of sidechain SMILES
                    "combination": "N/A"  # Placeholder
                }
                results.append(row)

            # Convert results to a DataFrame
            df_combinations = pd.DataFrame(results,
                                           columns=["ring_id", "ring_smiles", "sub_id", "sub_smiles", "combination"])

            # Append to Parquet file manually
            if not df_combinations.empty:
                # Read the existing Parquet file
                existing_table = pq.read_table(self.output_file)

                # Convert DataFrame to Arrow Table with the same schema
                new_table = pa.Table.from_pandas(df_combinations, schema=parquet_schema)

                # Concatenate existing and new tables
                combined_table = pa.concat_tables([existing_table, new_table])

                # Write back to Parquet file
                pq.write_table(combined_table, self.output_file)

            print(f"Processed ring {ring_id} with {n_subs} substituent positions.")

    def close_connection(self):
        """
        Closes the DuckDB connection.
        """
        self.con.close()
    
# Example usage
if __name__ == "__main__":
    # Define file paths
    HOME_DIR = Path.cwd()
    ring_file = HOME_DIR / "data" / "ring_list.parquet"
    sidechain_file = HOME_DIR / "data" / "sidechain_list.parquet"
    output_dir = HOME_DIR / "output"  # Store files in an output directory

    # Initialize and run the class
    generator = GenComs_Ring_Sidechains(ring_file, sidechain_file, output_dir)
    generator.load_data()
    generator.generate_combinations()
    generator.close_connection()


Initializing C:\Users\zachg\PycharmProjects\BillionMolecules\billion_mol_db\testing_recombination\output\combinations.parquet with an empty valid schema.
Processed ring R1 with 2 substituent positions.
Processed ring R2 with 3 substituent positions.
Processed ring R3 with 1 substituent positions.
Processed ring R3 with 5 substituent positions.
Processed ring R3 with 4 substituent positions.
Processed ring R3 with 4 substituent positions.
Processed ring R3 with 4 substituent positions.
Processed ring R3 with 4 substituent positions.
Processed ring R3 with 4 substituent positions.
Processed ring R3 with 4 substituent positions.


In [80]:
import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import itertools
import os


class GenComs_Ring_Sidechains:
    def __init__(self, ring_file, sidechain_file, output_dir):
        """
        Initializes the class with input ring and sidechain datasets and the output directory.
        The Parquet file name will be automatically determined to avoid overwrites.
        """
        self.ring_file = ring_file
        self.sidechain_file = sidechain_file
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.output_file = self.get_unique_filename()
        self.con = duckdb.connect()

    def get_unique_filename(self):
        """
        Generate a unique Parquet filename by appending _1, _2, etc., if a file already exists.
        """
        base_filename = "combinations"
        extension = ".parquet"
        output_path = self.output_dir / f"{base_filename}{extension}"

        counter = 1
        while output_path.exists():
            output_path = self.output_dir / f"{base_filename}_{counter}{extension}"
            counter += 1

        return output_path

    def load_data(self):
        """
        Loads ring and sidechain datasets into DuckDB tables.
        """
        self.con.execute(f"CREATE TABLE rings AS SELECT * FROM '{self.ring_file}'")
        self.con.execute(f"CREATE TABLE sidechains AS SELECT * FROM '{self.sidechain_file}'")

    def ensure_valid_parquet(self):
        """
        Ensures that the output Parquet file is valid by creating an empty file with the correct schema if needed.
        """
        parquet_schema = pa.schema([
            ("ring_id", pa.string()),
            ("ring_smiles", pa.string()),
            ("sub_id", pa.list_(pa.string())),  # Ordered list of sidechain IDs
            ("sub_smiles", pa.list_(pa.string())),  # Ordered list of sidechain SMILES
            ("combination", pa.string())  # Placeholder for now
        ])

        # If the file does not exist or is empty, create a valid empty Parquet file
        if not os.path.exists(self.output_file) or os.path.getsize(self.output_file) == 0:
            print(f"Initializing {self.output_file} with an empty valid schema.")
            empty_table = pa.Table.from_pandas(pd.DataFrame(columns=parquet_schema.names), schema=parquet_schema)
            pq.write_table(empty_table, self.output_file)

    def generate_combinations(self):
        """
        Processes rings one at a time, generating all possible sidechain combinations
        (including repeated sidechains) based on the number of substitution points.
        Saves results to the Parquet file iteratively.
        """
        # Ensure output file is valid before writing
        self.ensure_valid_parquet()

        # Read rings and sidechains tables into DuckDB
        rings_df = self.con.execute("SELECT * FROM rings").fetchdf()
        sidechains_df = self.con.execute("SELECT * FROM sidechains").fetchdf()

        # Define the output Parquet schema
        parquet_schema = pa.schema([
            ("ring_id", pa.string()),
            ("ring_smiles", pa.string()),
            ("sub_id", pa.list_(pa.string())),  # Ordered list of sidechain IDs
            ("sub_smiles", pa.list_(pa.string())),  # Ordered list of sidechain SMILES
            ("combination", pa.string())  # Placeholder for now
        ])

        # Process one ring at a time
        for _, ring in rings_df.iterrows():
            ring_id = ring["id"]
            ring_smiles = ring["smiles"]
            n_subs = ring["n_subs"]

            # Generate all possible combinations of `n_subs` sidechains (including repetitions)
            sidechain_combinations = list(itertools.product(sidechains_df.itertuples(index=False), repeat=n_subs))

            results = []
            for sidechain_set in sidechain_combinations:
                sub_ids = [sc.id for sc in sidechain_set]
                sub_smiles = [sc.smiles for sc in sidechain_set]

                # Construct a row with ordered lists
                row = {
                    "ring_id": str(ring_id),
                    "ring_smiles": str(ring_smiles),
                    "sub_id": sub_ids,  # List of sidechain IDs
                    "sub_smiles": sub_smiles,  # List of sidechain SMILES
                    "combination": "N/A"  # Placeholder
                }
                results.append(row)

            # Convert results to a DataFrame
            df_combinations = pd.DataFrame(results,
                                           columns=["ring_id", "ring_smiles", "sub_id", "sub_smiles", "combination"])

            # Append to Parquet file manually
            if not df_combinations.empty:
                # Read the existing Parquet file
                existing_table = pq.read_table(self.output_file)

                # Convert DataFrame to Arrow Table with the same schema
                new_table = pa.Table.from_pandas(df_combinations, schema=parquet_schema)

                # Concatenate existing and new tables
                combined_table = pa.concat_tables([existing_table, new_table])

                # Write back to Parquet file
                pq.write_table(combined_table, self.output_file)

            print(f"Processed ring {ring_id} with {n_subs} substituent positions.")

    def close_connection(self):
        """
        Closes the DuckDB connection.
        """
        self.con.close()


# Example usage
if __name__ == "__main__":
    # Define file paths
    HOME_DIR = Path.cwd()
    ring_file = HOME_DIR / "data" / "ring_list.parquet"
    sidechain_file = HOME_DIR / "data" / "sidechain_list.parquet"
    output_dir = HOME_DIR / "output"  # Store files in an output directory

    # Initialize and run the class
    generator = GenComs_Ring_Sidechains(ring_file, sidechain_file, output_dir)
    generator.load_data()
    generator.generate_combinations()
    generator.close_connection()

Initializing C:\Users\zachg\PycharmProjects\BillionMolecules\billion_mol_db\testing_recombination\output\combinations_6.parquet with an empty valid schema.
Processed ring R1 with 2 substituent positions.
Processed ring R2 with 3 substituent positions.
Processed ring R3 with 1 substituent positions.
Processed ring R3 with 5 substituent positions.
Processed ring R3 with 4 substituent positions.
Processed ring R3 with 4 substituent positions.
Processed ring R3 with 4 substituent positions.
Processed ring R3 with 4 substituent positions.
Processed ring R3 with 4 substituent positions.
Processed ring R3 with 4 substituent positions.


In [ ]:
import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import itertools
import multiprocessing
import time
from pathlib import Path


class GenComs_Ring_Sidechains:
    def __init__(self, ring_file, sidechain_file, output_dir):
        self.ring_file = ring_file
        self.sidechain_file = sidechain_file
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.output_file = self.get_unique_filename()
        self.con = duckdb.connect()

    def get_unique_filename(self):
        base_filename = "combinations"
        extension = ".parquet"
        output_path = self.output_dir / f"{base_filename}{extension}"

        counter = 1
        while output_path.exists():
            output_path = self.output_dir / f"{base_filename}_{counter}{extension}"
            counter += 1

        return output_path

    def load_data(self):
        """Loads ring and sidechain datasets into DuckDB tables."""
        self.con.execute(f"CREATE TABLE rings AS SELECT * FROM read_parquet('{self.ring_file}')")

    def ensure_valid_parquet(self):
        """Ensures that the output Parquet file has a valid schema."""
        parquet_schema = pa.schema([
            ("ring_id", pa.string()),
            ("ring_smiles", pa.string()),
            ("sub_id", pa.list_(pa.string())),
            ("sub_smiles", pa.list_(pa.string())),
            ("combination", pa.string())
        ])

        if not self.output_file.exists() or self.output_file.stat().st_size == 0:
            print(f"Initializing {self.output_file} with an empty valid schema.")
            empty_table = pa.Table.from_pandas(pd.DataFrame(columns=parquet_schema.names), schema=parquet_schema)
            pq.write_table(empty_table, self.output_file)

    def generate_combinations(self, ring_batch, proc_id):
        """Each processor runs only its assigned batch of rings."""
        start_time_proc = time.time()

        # Connect to DuckDB in this process
        con = duckdb.connect()
        sidechains_df = con.execute(f"SELECT * FROM read_parquet('{self.sidechain_file}')").fetchdf()
        con.close()

        parquet_schema = pa.schema([
            ("ring_id", pa.string()),
            ("ring_smiles", pa.string()),
            ("sub_id", pa.list_(pa.string())),
            ("sub_smiles", pa.list_(pa.string())),
            ("combination", pa.string())
        ])

        print(f"Processor {proc_id} assigned rings: {[ring['id'] for ring in ring_batch]}")

        for ring in ring_batch:
            start_time_ring = time.time()
            ring_id = ring["id"]
            ring_smiles = ring["smiles"]
            n_subs = int(ring["n_subs"])

            sidechain_combinations = list(itertools.product(sidechains_df.itertuples(index=False), repeat=n_subs))

            results = []
            for sidechain_set in sidechain_combinations:
                sub_ids = [sc.id for sc in sidechain_set]
                sub_smiles = [sc.smiles for sc in sidechain_set]

                results.append({
                    "ring_id": str(ring_id),
                    "ring_smiles": str(ring_smiles),
                    "sub_id": sub_ids,
                    "sub_smiles": sub_smiles,
                    "combination": "N/A"
                })

            df_combinations = pd.DataFrame(results, columns=["ring_id", "ring_smiles", "sub_id", "sub_smiles", "combination"])

            if not df_combinations.empty:
                existing_table = pq.read_table(self.output_file)
                new_table = pa.Table.from_pandas(df_combinations, schema=parquet_schema)
                combined_table = pa.concat_tables([existing_table, new_table])
                pq.write_table(combined_table, self.output_file)

            end_time_ring = time.time()
            print(f"Processor {proc_id} ran ring {ring_id} for {end_time_ring - start_time_ring:.2f}s")

        end_time_proc = time.time()
        print(f"Processor {proc_id} finished its batch in {end_time_proc - start_time_proc:.2f}s")

    def run_parallel_processing(self):
        """Splits rings among processors explicitly and runs in parallel."""
        rings_df = self.con.execute("SELECT * FROM rings").fetchdf()
        self.con.close()  # Close DuckDB since it's not used in child processes

        # Get number of processors
        n_processors = min(multiprocessing.cpu_count(), len(rings_df))
        print(f"Using {n_processors} processors for {len(rings_df)} rings")

        # Manually assign rings to processors
        ring_batches = [rings_df.iloc[i::n_processors].to_dict(orient="records") for i in range(n_processors)]

        # Print the assignments before running
        for i, batch in enumerate(ring_batches):
            print(f"Processor {i} assigned {len(batch)} rings: {[ring['id'] for ring in batch]}")

        # Run multiprocessing
        with multiprocessing.Pool(n_processors) as pool:
            pool.starmap(self.generate_combinations, [(batch, i) for i, batch in enumerate(ring_batches)])


# Example usage
if __name__ == "__main__":
    HOME_DIR = Path.cwd()
    ring_file = HOME_DIR / "data" / "ring_list.parquet"
    sidechain_file = HOME_DIR / "data" / "sidechain_list.parquet"
    output_dir = HOME_DIR / "output"

    generator = GenComs_Ring_Sidechains(ring_file, sidechain_file, output_dir)
    generator.load_data()
    generator.ensure_valid_parquet()
    generator.run_parallel_processing()


In [ ]:
print("hello world")

In [72]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

# Define the path to the Parquet file
parquet_file = Path(r"C:\Users\zachg\PycharmProjects\BillionMolecules\billion_mol_db\testing_recombination\output\combinations_3.parquet")

# Check if the file exists
if not parquet_file.exists():
    print(f"Error: File {parquet_file} does not exist.")
else:
    # Read the Parquet file into a Pandas DataFrame
    df = pd.read_parquet(parquet_file, engine="pyarrow")

    # Print the DataFrame
    print("Loaded DataFrame from Parquet:")
    print(df)


Loaded DataFrame from Parquet:
Empty DataFrame
Columns: [ring_id, ring_smiles, sub_id, sub_smiles, combination]
Index: []
